## Load Dependencies

In [44]:
import os
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import SystemMessage, trim_messages
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
load_dotenv()

True

## Model Setup

In [ ]:
## Load API key
groq_api_key = os.getenv("GROQ_API_KEY")

In [4]:
## Load Model
model = ChatGroq(model="gemma2-9b-it", groq_api_key=groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x0000029FFD047100>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000029FFD0815A0>, model_name='gemma2-9b-it', model_kwargs={}, groq_api_key=SecretStr('**********'))

## Try Model

In [5]:
model.invoke([HumanMessage(content="Hi, I am Shreya Gautam and I am a Computer Science Master's Student at Polimi.")])

AIMessage(content="Hi Shreya Gautam! \n\nIt's nice to meet you. Being a Computer Science Master's student at Polimi is impressive! \n\nIs there anything you'd like to talk about or ask me? I'm here to help with any questions you might have or just have a conversation.\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 30, 'total_tokens': 96, 'completion_time': 0.12, 'prompt_time': 0.001400859, 'queue_time': 0.188288596, 'total_time': 0.121400859}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--1794eb3a-8d22-4820-a2c0-197b3cb9e52f-0', usage_metadata={'input_tokens': 30, 'output_tokens': 66, 'total_tokens': 96})

In [7]:
## Checking if the model is able to answer from history
model.invoke(
    [
        HumanMessage(content="Hi, I am Shreya Gautam and I am a Computer Science Master's Student at Polimi."),
        AIMessage(content="Hi Shreya Gautam! \n\nIt's nice to meet you. Being a Computer Science Master's student at Polimi is impressive! \n\nIs there anything you'd like to talk about or ask me? I'm here to help with any questions you might have or just have a conversation.\n"),
        HumanMessage(content="Tell me what's my name and what do I do.")
    ]
)

AIMessage(content="You are Shreya Gautam, and you are a Computer Science Master's student at Polimi.  \n\nIs there anything else you'd like to know or discuss? 😊 \n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 117, 'total_tokens': 157, 'completion_time': 0.072727273, 'prompt_time': 0.003669118, 'queue_time': 0.187621697, 'total_time': 0.076396391}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--48c11e89-14e2-4136-addd-041969bc30be-0', usage_metadata={'input_tokens': 117, 'output_tokens': 40, 'total_tokens': 157})

We can see that in the list of messages, the model is able to remmeber the history.

## Message History
We can use `Message History class` to wrap our model and make it `stateful`. This helps in keeping track of input and outputs of the model and store them in some datastore. Any further interactions in future will first load those messages and then pass them into the chain as part of the input.

In [20]:
store={}
def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

We are storing the `session_id` in `store` dictionary. With every `session_id` we are initialising a `chat_message_histories` (which means it is an object of chat_message_histories)

In [21]:
config={"configurable":{"session_id":"chat1"}}

In [23]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi, I am Shreya Gautam and I am a Computer Science Master's Student at Polimi.")],
    config=config
)

In [24]:
response.content

"Hi Shreya! \n\nIt's great to meet you.  Polimi is a fantastic university for Computer Science. What area of Computer Science are you specializing in for your Master's? \n\nI'm always interested in learning about what students are working on.\n"

In [25]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config
    )

AIMessage(content='Your name is Shreya Gautam. 😊 \n\nI remembered it from our earlier conversation!  \n\n\n\nHow can I help you today?\n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 222, 'total_tokens': 252, 'completion_time': 0.054545455, 'prompt_time': 0.004868598, 'queue_time': 0.189452732, 'total_time': 0.059414053}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--c70277b5-fdc9-481e-b39f-3bc5948de35f-0', usage_metadata={'input_tokens': 222, 'output_tokens': 30, 'total_tokens': 252})

We can see that it is able to remember me. \
Now if we change the config (i.e., change the session_id), will it be able to remember the context?

In [26]:
## changing the config --> session_id
config1={"configurable":{"session_id":"chat2"}}
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config1
)
response.content

"As an AI, I have no memory of past conversations and no access to personal information about you. So, I don't know your name. 😊\n\nIf you'd like to tell me, I'm happy to know!\n"

Therefore, we can clearly see that with the help of `session_id` we are easily able to switch the context of the conversation

## Prompt Templates
These help to turn raw user information into a LLM compatible format.\
Until now we were just taking the raw message and passing to the LLM. Now we will add in a system message with some custom instruction, taking messages as input along with some more inputs.

#### Experiment1

In [29]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You're a smart, honest, and concise assistant who answers user questions clearly and helpfully."),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt|model

Now we need to give the messages in a key-value pair because of MessagePlaceholder


In [30]:
chain.invoke({"messages":[HumanMessage(content="Hi, my name is Shreya!")]})

AIMessage(content="Hi Shreya, it's nice to meet you!  \n\nHow can I help you today? 😊 \n\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 37, 'total_tokens': 64, 'completion_time': 0.049090909, 'prompt_time': 0.00155399, 'queue_time': 0.186657248, 'total_time': 0.050644899}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--1dfe22c1-897b-49d7-afde-6655d83fabcf-0', usage_metadata={'input_tokens': 37, 'output_tokens': 27, 'total_tokens': 64})

Now we will see invoking with respect to the chat message history

In [31]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

In [33]:
config = {"configurable":{"session_id":"chat3"}}
response = with_message_history.invoke(
    [HumanMessage(content="Hi, my name is Shreya!")],
    config=config
)
response.content

"Hello Shreya! It's nice to meet you. 😊 \n\nWhat can I help you with today?  \n"

#### Experiment2 (adding language to the prompt)

In [34]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You're a smart, honest, and concise assistant who answers user questions clearly and helpfully in {language}"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt|model

In [35]:
response = chain.invoke(
    {
        "messages":[HumanMessage(content="Hi, my name is Shreya!")],
        "language": "Hindi"
    }
)
response.content

'नमस्ते श्रेया! \n\nमुझे आपसे मिलकर खुशी हो रही है। मैं आपकी मदद करने के लिए तैयार हूँ।  आप मुझसे क्या पूछना चाहेंगी? 😊  \n'

In [36]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [37]:
config = {"configurable":{"session_id":"chat4"}}
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="Hi, my name is Shreya!")],
        "language": "Hindi"
    },
    config=config
)
response.content

'नमस्ते श्रेया! 👋 \n\nमुझे बहुत खुशी है आपसे मिलने की।  आप मुझसे क्या जानना चाहती हैं? 😊 \n\n'

In [38]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="What's my name?")],
        "language": "Hindi"
    },
    config=config,
)
response.content

'आपका नाम श्रेया है। 😊  \n'

## Managing the Conversation History
The conversation history needs to be managed otherwise, the list of messages will grow unbounded and potentially overflo the context window of the LLM. Thus managing means limiting the size of the messages being passed.

we will be using `trim_messages` which helps to reduce the number of messages being sent to the model. It allows us to specify the  umber of tokens we want to keep along with other parameters 

In [43]:
trimmer = trim_messages(
    max_tokens=40,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

messages = [
    SystemMessage(content="You're a good assistant"),
    HumanMessage(content="Hi, I am Gautam"),
    AIMessage(content="Hi"),
    HumanMessage(content="I like basil gelato"),
    AIMessage(content="nice"),
    HumanMessage(content="What is 2+2 ?"),
    AIMessage(content="4"),
    HumanMessage(content="Thanks!"),
    AIMessage(content="No Problem!"),
    HumanMessage(content="Having fun?"),
    AIMessage(content="yes!")
]

trimmer.invoke(messages)

[SystemMessage(content="You're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is 2+2 ?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Thanks!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='No Problem!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={})]

In [45]:
chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
)

response = chain.invoke(
    {
        "messages":messages + [HumanMessage(content="What gelato do I like?")],
        "language": "English"
    }
)

response.content

"As an AI, I don't have personal preferences or memories, so I don't know what gelato you like!  \n\nWhat are some of your favorite flavors? Maybe I can help you find a new one to try. 😋🍦  \n\n\n\n\n"

It isn't able to find my gelato flavor because the trimmer trimmed of that particular context.

In [48]:
response = chain.invoke(
    {
        "messages":messages + [HumanMessage(content="What all questions did I ask?")],
        "language": "English"
    }
)

response.content

'You asked me:\n\n1.  "You\'re a smart, honest, and concise assistant who answers user questions clearly and helpfully in English You\'re a good assistant Thanks!" (This was more of a statement but also a question about my abilities)\n2. "Having fun?"\n3. "What all questions did I ask?" \n\n\nLet me know if you have any other questions!\n'

#### Wrap this in Message History

In [49]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)
config = {"configurable":{"session_id":"chat5"}}

In [55]:
response = chain.invoke(
    {
        "messages":messages + [HumanMessage(content="What questions did I ask you?")],
        "language": "English"
    },
    config=config
)

response.content

'You asked:\n\n1. "You\'re a smart, honest, and concise assistant who answers user questions clearly and helpfully in English. You\'re a good assistant Thanks!"\n2. "Having fun?" \n3. "What questions did I ask you?" \n\n\n\n'